<a href="https://colab.research.google.com/github/Anupampal1992/Group-Project/blob/EDA/Traffic_Accident.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [189]:
!git clone  https://github.com/Anupampal1992/Group-Project.git
%cd Group-Project
!ls

Cloning into 'Group-Project'...
remote: Enumerating objects: 48, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 48 (delta 29), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (48/48), 207.03 KiB | 3.98 MiB/s, done.
Resolving deltas: 100% (29/29), done.
/content/Group-Project/Group-Project/Group-Project/Group-Project/Group-Project/Group-Project/Group-Project
 Traffic_Accident.ipynb  'Traffic Accident Severity Predictor Dataset.csv'


In [190]:

!git status


On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [191]:
import pandas as pd
import numpy as np

#visualization
import matplotlib.pyplot as plt
import seaborn as sns

#preprocessing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from scipy.spatial import transform


#Models
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.metrics import f1_score

#Evaluation
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, roc_auc_score,accuracy_score

#Pipeline
from sklearn.pipeline import Pipeline

#Compute Class Weight
from sklearn.utils.class_weight import compute_class_weight



In [192]:
df=pd.read_csv('Traffic Accident Severity Predictor Dataset.csv')  #Loading the data from Github
df.head(5)

,Weather,Road_Type,Time_of_Day,Traffic_Density,Speed_Limit,Number_of_Vehicles,Driver_Alcohol,Accident_Severity,Road_Condition,Vehicle_Type,Driver_Age,Driver_Experience,Road_Light_Condition,Accident
0,Rainy,City Road,Morning,1.0,100.0,5.0,0.0,NaN,Wet,Car,51.0,48.0,Artificial Light,0
1,Clear,Rural Road,Night,NaN,120.0,3.0,0.0,Moderate,Wet,Truck,49.0,43.0,Artificial Light,0
2,Rainy,Highway,Evening,1.0,60.0,4.0,0.0,Low,Icy,Car,54.0,52.0,Artificial Light,0
3,Clear,City Road,Afternoon,2.0,60.0,3.0,0.0,Low,Under Construction,Bus,34.0,31.0,Daylight,0
4,Rainy,Highway,Morning,1.0,195.0,11.0,0.0,Low,Dry,Car,62.0,55.0,Artificial Light,1


EDA

In [193]:
df.shape

(20000, 14)

In [194]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Weather               19958 non-null  object 
 1   Road_Type             19958 non-null  object 
 2   Time_of_Day           19958 non-null  object 
 3   Traffic_Density       19958 non-null  float64
 4   Speed_Limit           19958 non-null  float64
 5   Number_of_Vehicles    19958 non-null  float64
 6   Driver_Alcohol        19958 non-null  float64
 7   Accident_Severity     19958 non-null  object 
 8   Road_Condition        19958 non-null  object 
 9   Vehicle_Type          19958 non-null  object 
 10  Driver_Age            19958 non-null  float64
 11  Driver_Experience     19958 non-null  float64
 12  Road_Light_Condition  19958 non-null  object 
 13  Accident              20000 non-null  int64  
dtypes: float64(6), int64(1), object(7)
memory usage: 2.1+ MB


In [195]:
df.describe()

,Traffic_Density,Speed_Limit,Number_of_Vehicles,Driver_Alcohol,Driver_Age,Driver_Experience,Accident
count,19958.000000,19958.000000,19958.000000,19958.000000,19958.000000,19958.000000,20000.000000
mean,1.010923,71.448943,3.282694,0.164445,43.146758,38.859154,0.292000
std,0.783966,32.366260,1.999111,0.370688,15.099349,15.249536,0.454694
min,0.000000,30.000000,1.000000,0.000000,18.000000,9.000000,0.000000
25%,0.000000,50.000000,2.000000,0.000000,30.000000,26.000000,0.000000
50%,1.000000,60.000000,3.000000,0.000000,43.000000,39.000000,0.000000
75%,2.000000,80.000000,4.000000,0.000000,56.000000,52.000000,1.000000
max,2.000000,213.000000,14.000000,1.000000,69.000000,69.000000,1.000000


In [196]:
df['Accident_Severity'].value_counts()  #Target variable

,count
Accident_Severity,
Low,11862
Moderate,6090
High,2006


In [197]:
df.isnull().sum()   #cheacking the null value

,0
Weather,42
Road_Type,42
Time_of_Day,42
Traffic_Density,42
Speed_Limit,42
Number_of_Vehicles,42
Driver_Alcohol,42
Accident_Severity,42
Road_Condition,42
Vehicle_Type,42


In [198]:
#Separating the numerical and categorical columns
num_col=df.select_dtypes(include=['int64','float64']).columns
cat_col=df.select_dtypes(include=['object']).columns

In [207]:
# Fillig the missing values
df[num_col] = df[num_col].fillna(df[num_col].median())
df[cat_col] = df[cat_col].fillna(df[cat_col].mode().iloc[0])   #.mode() returns a DataFrame (or Series with multiple values), but fillna() needs a single value per column.
print("Numerical Coulumns", df[num_col].columns)
print("Categorical Columns", df[cat_col].columns)
df.isnull().sum()

Numerical Coulumns Index(['Traffic_Density', 'Speed_Limit', 'Number_of_Vehicles',
       'Driver_Alcohol', 'Driver_Age', 'Driver_Experience', 'Accident'],
      dtype='object')
Categorical Columns Index(['Weather', 'Road_Type', 'Time_of_Day', 'Accident_Severity',
       'Road_Condition', 'Vehicle_Type', 'Road_Light_Condition'],
      dtype='object')


,0
Weather,0
Road_Type,0
Time_of_Day,0
Traffic_Density,0
Speed_Limit,0
Number_of_Vehicles,0
Driver_Alcohol,0
Accident_Severity,0
Road_Condition,0
Vehicle_Type,0


**Feature and target separation**

In [200]:
x=df.drop(columns=['Accident_Severity', 'Accident'], axis=1)
y=df['Accident_Severity']
print(x.shape)
print(y.shape)
x.columns

(20000, 12)
(20000,)


Index(['Weather', 'Road_Type', 'Time_of_Day', 'Traffic_Density', 'Speed_Limit',
       'Number_of_Vehicles', 'Driver_Alcohol', 'Road_Condition',
       'Vehicle_Type', 'Driver_Age', 'Driver_Experience',
       'Road_Light_Condition'],
      dtype='object')

In [201]:
y.count()

np.int64(19959)

In [202]:

print("Clases of Target Variable:", y.value_counts())


Clases of Target Variable: Accident_Severity
Low         11863
Moderate     6090
High         2006
Name: count, dtype: int64


In [203]:
order = {'Low': 0, 'Moderate': 1, 'High': 2}
y = y.map(order)
y

,Accident_Severity
0,0.0
1,1.0
2,0.0
3,0.0
4,0.0
...,...
19995,1.0
19996,1.0
19997,1.0
19998,0.0


In [204]:
#get the numerical features and categorical features without Target variable
numerical_features = x.select_dtypes(include=['int64', 'float64']).columns
categorical_features = x.select_dtypes(include=['object']).columns
print("Numerical Features:", numerical_features)
print("Categorical Features:", categorical_features)

Numerical Features: Index(['Traffic_Density', 'Speed_Limit', 'Number_of_Vehicles',
       'Driver_Alcohol', 'Driver_Age', 'Driver_Experience'],
      dtype='object')
Categorical Features: Index(['Weather', 'Road_Type', 'Time_of_Day', 'Road_Condition', 'Vehicle_Type',
       'Road_Light_Condition'],
      dtype='object')


In [205]:
# Apply different processing step to different columns group
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])
categorical_features


Index(['Weather', 'Road_Type', 'Time_of_Day', 'Road_Condition', 'Vehicle_Type',
       'Road_Light_Condition'],
      dtype='object')

**Split and Train the data**

In [206]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42,  stratify=y
)

print("Shape of x_train:", x_train.shape)
print("Shape of x_test:", x_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)
y_train.value_counts()

ValueError: Input y contains NaN.

**Applying Models**

In [ ]:
#Logistic regression
lr = Pipeline(steps=[('preprocessor', preprocessor),
                     ('classifier', LogisticRegression(max_iter=1000))])

lr=lr.fit(x_train, y_train)
y_pred = lr.predict(x_test)

print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("accuracy:\n", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

In [ ]:
#Random Forest

rf = Pipeline(steps=[('preprocessor', preprocessor),
                      ('model', RandomForestClassifier(
                          class_weight='balanced_subsample',
                          n_estimators=300,
                          max_depth=None,
                          min_samples_split=5,
                          min_samples_leaf=2,
                          random_state=42,

                      ))])

rf = rf.fit(x_train, y_train)
y_pred = rf.predict(x_test)

print("confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("accuracy:\n", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))


**Compute class weights**

In [ ]:
classes = np.unique(y_train)
class_weights = compute_class_weight(
    class_weight='balanced', classes=classes, y=y_train
)

weight_map = dict(zip(classes, class_weights))
sample_weights = y_train.map(weight_map)
sample_weights

In [ ]:
# XGBoost Model
xgb = Pipeline(steps=[('preprocessor', preprocessor),
                      ('classifier', XGBClassifier(
                          objective='multi:softmax',
                         num_class=3,
                          eval_metric='mlogloss',
                          max_depth=5,
                          sub_sample=0.8,
                          colsample_bytree=0.8,

                          n_estimators=300,
                          learning_rate=0.05
                      ))])

xgb.fit(x_train, y_train, classifier__sample_weight=sample_weights)
y_pred = xgb.predict(x_test)
y_prob = xgb.predict_proba(x_test)
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("accuracy:\n", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

**Applying threshold**

In [ ]:
for t in [0.4,0.5,0.6,0.7,0.8,0.9,0.95]:
    weights = np.array([1, 1, t])
    preds = np.argmax(y_prob / weights, axis=1)
    print(f"\nThreshold = {t}")
    print(("F1 Score:"),f1_score(y_test, preds, average='weighted'))
    print(("Accuracy_metrics:"),accuracy_score(y_test, preds))
    print(confusion_matrix(y_test, preds))
    print(classification_report(y_test, preds))


In [ ]:
FINAL_THRESHOLD = .8

y_pred_final = np.argmax(y_prob / [1, 1, FINAL_THRESHOLD], axis=1)

print(f1_score(y_test, y_pred_final, average='weighted'))
print(confusion_matrix(y_test, y_pred_final))
print(accuracy_score(y_test, y_pred_final))
print(classification_report(y_test, y_pred_final))

**Feature Importance**

In [ ]:
#Get the trained XGBoost model from Pipeline
xgb_model = xgb.named_steps['classifier']

#Extract feature names after preprocessing
preprocessor = xgb.named_steps['preprocessor']
numerical_features = preprocessor.transformers_[0][2]
categorical_features = preprocessor.transformers_[1][2]
ohe = preprocessor.transformers_[1][1]
categorical_features = ohe.get_feature_names_out(categorical_features)
feature_names = list(numerical_features) + list(categorical_features)

len(feature_names) == len(xgb_model.feature_importances_)

**Create Feature Importance DataFrame**

In [ ]:
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': xgb_model.feature_importances_
})

importance_df = importance_df.sort_values(by='Importance', ascending=False)
importance_df

In [ ]:
#View Top Important Features

importance_df.head(10)